In [0]:
from pyspark.sql import functions as F

# Lire la première ligne en texte brut pour voir le vrai séparateur
with open("/Volumes/workspace/default/raw_data/sirene/data.csv", "r", encoding="utf-8") as f:
    premiere_ligne = f.readline()

print(premiere_ligne)

﻿SIREN;NIC;SIRET;Statut de diffusion de l'établissement;Date de création de l'établissement;Tranche de l'effectif de l'établissement;Tranche de l'effectif de l'établissement triable;Année de la tranche d'effectif de l'établissement;Activité principale de l'établissement;Date de la dernière mise à jour de l'établissement;Etablissement siège;Nombre de periodes de l'établissement;Complément d'adresse de l'établissement;Numéro de voie de l'établissement;Indice de répétition de l'établissement;Type de voie de l'établissement;Libellé de la voie de l'établissement;Code postal de l'établissement;Commune de l'établissement;Libellé de la commune de l'établissement à l'étranger;Distribution spéciale de l'établissement;Code commune de l'établissement;Code cedex de l'établissement;Libellé cedex de l'établissement;Code du pays de l'établissement étranger;Libellé du pays de l'établissement étranger;Complément d'adresse de l'établissement 2;Numero de voie de l'établissement 2;Indice de répétition de l

In [0]:
SEP = ";"

df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", SEP)
    .option("encoding", "UTF-8")
    .csv("/Volumes/workspace/default/raw_data/sirene/data.csv")
)

print(f"Nombre de colonnes : {len(df_raw.columns)}")
print(f"Nombre de lignes   : {df_raw.count()}")

Nombre de colonnes : 107
Nombre de lignes   : 420411


In [0]:
for i, col_name in enumerate(df_raw.columns):
    print(f"{i:3d} | {col_name}")

  0 | SIREN
  1 | NIC
  2 | SIRET
  3 | Statut de diffusion de l'établissement
  4 | Date de création de l'établissement
  5 | Tranche de l'effectif de l'établissement
  6 | Tranche de l'effectif de l'établissement triable
  7 | Année de la tranche d'effectif de l'établissement
  8 | Activité principale de l'établissement8
  9 | Date de la dernière mise à jour de l'établissement
 10 | Etablissement siège
 11 | Nombre de periodes de l'établissement
 12 | Complément d'adresse de l'établissement
 13 | Numéro de voie de l'établissement
 14 | Indice de répétition de l'établissement
 15 | Type de voie de l'établissement
 16 | Libellé de la voie de l'établissement
 17 | Code postal de l'établissement
 18 | Commune de l'établissement
 19 | Libellé de la commune de l'établissement à l'étranger
 20 | Distribution spéciale de l'établissement
 21 | Code commune de l'établissement
 22 | Code cedex de l'établissement
 23 | Libellé cedex de l'établissement
 24 | Code du pays de l'établissement étrang

In [0]:
rename_map = {
    "SIREN": "siren",
    "NIC": "nic",
    "SIRET": "siret",
    "Statut de diffusion de l'établissement": "statut_diffusion",
    "Date de création de l'établissement": "date_creation_etab",
    "Tranche de l'effectif de l'établissement": "tranche_effectif",
    "Activité principale de l'établissement8": "activite_principale_etab",
    "Etablissement siège": "etablissement_siege",
    "Code postal de l'établissement": "code_postal",
    "Commune de l'établissement": "commune",
    "Code commune de l'établissement": "code_commune",
    "Code du département de l'établissement": "code_departement",
    "Département de l'établissement": "departement",
    "Code de la région de l'établissement": "code_region",
    "Région de l'établissement": "region",
    "Etat administratif de l'établissement": "etat_admin_etab",
    "Date de fermeture de l'établissement": "date_fermeture_etab",
    "Dénomination de l'unité légale": "denomination_unite_legale",
    "Catégorie de l'entreprise": "categorie_entreprise",
    "Etat administratif de l'unité légale": "etat_admin_ul",
    "Caractère employeur de l'unité légale": "caractere_employeur",
    "Activité principale de l'unité légale": "activite_principale_ul",
    "Catégorie juridique de l'unité légale": "categorie_juridique",
    "Date de création de l'unité légale": "date_creation_ul",
}

# Appliquer le renommage avec garde-fou : signale toute colonne introuvable
df_renamed = df_raw
for old_name, new_name in rename_map.items():
    if old_name in df_renamed.columns:
        df_renamed = df_renamed.withColumnRenamed(old_name, new_name)
    else:
        print(f"Colonne introuvable : {old_name}")

df_renamed.select(list(rename_map.values())).printSchema()

root
 |-- siren: integer (nullable = true)
 |-- nic: integer (nullable = true)
 |-- siret: long (nullable = true)
 |-- statut_diffusion: string (nullable = true)
 |-- date_creation_etab: string (nullable = true)
 |-- tranche_effectif: string (nullable = true)
 |-- activite_principale_etab: string (nullable = true)
 |-- etablissement_siege: string (nullable = true)
 |-- code_postal: string (nullable = true)
 |-- commune: string (nullable = true)
 |-- code_commune: integer (nullable = true)
 |-- code_departement: integer (nullable = true)
 |-- departement: string (nullable = true)
 |-- code_region: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- etat_admin_etab: string (nullable = true)
 |-- date_fermeture_etab: date (nullable = true)
 |-- denomination_unite_legale: string (nullable = true)
 |-- categorie_entreprise: string (nullable = true)
 |-- etat_admin_ul: string (nullable = true)
 |-- caractere_employeur: string (nullable = true)
 |-- activite_principale_ul: st

In [0]:
import time
from pyspark.sql import functions as F

# Transformation seule - instantané, rien n'est calculé
debut = time.time()
df_test = df_renamed.filter(F.col("etat_admin_etab") == "Actif").select("siren", "commune")
print(f"Transformation déclarée en {time.time() - debut:.4f} sec")

# Action - déclenche le calcul réel
debut = time.time()
nb = df_test.count()
print(f"Action (count) exécutée en {time.time() - debut:.4f} sec → {nb} lignes")

Transformation déclarée en 0.0003 sec
Action (count) exécutée en 3.9907 sec → 162978 lignes


In [0]:
df_renamed.select("etat_admin_etab").distinct().show()
df_renamed.select("statut_diffusion").distinct().show()
df_renamed.select("etablissement_siege").distinct().show()

+---------------+
|etat_admin_etab|
+---------------+
|          Fermé|
|          Actif|
+---------------+

+----------------+
|statut_diffusion|
+----------------+
|               P|
|               O|
+----------------+

+-------------------+
|etablissement_siege|
+-------------------+
|                oui|
|                non|
+-------------------+



In [0]:
colonnes_a_nettoyer = [
    "denomination_unite_legale",
    "activite_principale_etab",
    "categorie_juridique",
    "categorie_entreprise",
]

df_clean_nd = df_renamed
for c in colonnes_a_nettoyer:
    df_clean_nd = df_clean_nd.withColumn(
        c,
        F.when((F.col(c) == "[ND]") | (F.col(c) == ""), None).otherwise(F.col(c))
    )

In [0]:
df_filtered = (
    df_clean_nd
    .filter(F.col("etat_admin_etab") == "Actif")
    .filter(F.col("statut_diffusion") != "P")
)

print(f"Lignes avant filtre : {df_clean_nd.count()}")
print(f"Lignes après filtre : {df_filtered.count()}")

Lignes avant filtre : 420411
Lignes après filtre : 134661


In [0]:
df_enriched = (
    df_filtered
    .withColumn("loaded_at", F.current_timestamp())
    .withColumn("siret_calcule", F.concat(F.col("siren"), F.col("nic")))
    .withColumn(
        "est_siege",
        F.when(F.col("etablissement_siege").isin("true", "oui", "True", "1"), True)
         .when(F.col("etablissement_siege").isin("false", "non", "False", "0"), False)
         .otherwise(None)
    )
)

df_enriched.select("siren", "siret_calcule", "est_siege", "loaded_at").show(5, truncate=False)

+---------+-------------+---------+--------------------------+
|siren    |siret_calcule|est_siege|loaded_at                 |
+---------+-------------+---------+--------------------------+
|808719801|80871980133  |true     |2026-07-06 14:32:14.489513|
|920181153|92018115315  |true     |2026-07-06 14:32:14.489513|
|892492018|89249201815  |true     |2026-07-06 14:32:14.489513|
|889847356|88984735617  |true     |2026-07-06 14:32:14.489513|
|501700744|50170074437  |true     |2026-07-06 14:32:14.489513|
+---------+-------------+---------+--------------------------+
only showing top 5 rows


In [0]:
# Répartition actif/fermé
df_enriched.groupBy("etat_admin_etab").count().show()

# Validation département - doit être très majoritairement 44
df_enriched.groupBy("code_departement").count().orderBy(F.desc("count")).show()

# Top 10 communes
df_enriched.groupBy("commune").count().orderBy(F.desc("count")).show(10, truncate=False)

+---------------+------+
|etat_admin_etab| count|
+---------------+------+
|          Actif|134661|
+---------------+------+

+----------------+------+
|code_departement| count|
+----------------+------+
|              44|134661|
+----------------+------+

+-------------------------+-----+
|commune                  |count|
+-------------------------+-----+
|NANTES                   |74724|
|SAINT-HERBLAIN           |10268|
|REZE                     |7159 |
|VERTOU                   |4819 |
|ORVAULT                  |4742 |
|CARQUEFOU                |4253 |
|SAINT-SEBASTIEN-SUR-LOIRE|3674 |
|BOUGUENAIS               |3258 |
|LA CHAPELLE-SUR-ERDRE    |3098 |
|COUERON                  |3040 |
+-------------------------+-----+
only showing top 10 rows


In [0]:
df_clean = df_enriched

print(f"Nombre de colonnes final : {len(df_clean.columns)}")
print(f"Nombre de lignes final   : {df_clean.count()}")
df_clean.printSchema()

Nombre de colonnes final : 110
Nombre de lignes final   : 134661
root
 |-- siren: integer (nullable = true)
 |-- nic: integer (nullable = true)
 |-- siret: long (nullable = true)
 |-- statut_diffusion: string (nullable = true)
 |-- date_creation_etab: string (nullable = true)
 |-- tranche_effectif: string (nullable = true)
 |-- Tranche de l'effectif de l'établissement triable: integer (nullable = true)
 |-- Année de la tranche d'effectif de l'établissement: integer (nullable = true)
 |-- activite_principale_etab: string (nullable = true)
 |-- Date de la dernière mise à jour de l'établissement: timestamp (nullable = true)
 |-- etablissement_siege: string (nullable = true)
 |-- Nombre de periodes de l'établissement: integer (nullable = true)
 |-- Complément d'adresse de l'établissement: string (nullable = true)
 |-- Numéro de voie de l'établissement: integer (nullable = true)
 |-- Indice de répétition de l'établissement: string (nullable = true)
 |-- Type de voie de l'établissement: stri